In [ ]:
import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))

import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file,path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)


# %%
import os
import gdsfactory as gf
from gdsfactory import Component
from glayout import MappedPDK, gf180
from glayout import nmos, pmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring
from glayout.primitives.mimcap import mimcap_array
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route
from glayout.util.comp_utils import evaluate_bbox, prec_center, prec_ref_center, align_comp_to_port
from glayout.util.port_utils import add_ports_perimeter,print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist


# Base parameters for PMOS and NMOS
stdp_config = {
    "pdk": gf180,
    "layout_rules": {
        "spacing": gf180.util_max_metal_seperation()+1,
        "routing_metal": "met2",
        "dummy_devices": False,
        "tie_layers": ("met2", "met1"),
        "sd_rmult": 1,
    },
}

nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": False,
    "with_substrate_tap":False
}

pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": False,
    "with_substrate_tap":False
}



In [ ]:
pdk = gf180
current_mirror = Component(name="current_mirror")
current_mirror.name = "current_mirror"
rules = stdp_config["layout_rules"]
spacing = rules["spacing"]
tie_layers = rules["tie_layers"]
sd_rmult = rules["sd_rmult"]


#Placement
m1 = nmos(pdk, width=4, length=0.28, with_dummy=(False, False),  tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
m2 = nmos(pdk, width=4, length=4, with_dummy=(False, False),  tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
m3 = pmos(pdk, width=8, length=0.5, with_dummy=(False, False),  tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
m4 = pmos(pdk, width=1.6, length=0.5, with_dummy=(False, False), tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)

m1.name = "M1" 
m2.name = "M2" 
m3.name = "M3" 
m4.name = "M4" 

m1_ref = current_mirror << m1
m2_ref = current_mirror << m2
m3_ref = current_mirror << m3
m4_ref = current_mirror << m4


bottom_row_y = evaluate_bbox(m1)[1]/2 + 4
top_row_y = bottom_row_y + evaluate_bbox(m3)[1]

m1_ref.move((evaluate_bbox(m1)[0]/2,bottom_row_y))
m2_ref.move((m1_ref.xmax + evaluate_bbox(m2_ref)[0]/2 + spacing , bottom_row_y))


m4_ref.move((evaluate_bbox(m1)[0]/2 ,top_row_y ))
m3_ref.move((m4_ref.xmax + evaluate_bbox(m3)[0]/2 + spacing,top_row_y))



In [ ]:
help(nmos)

In [ ]:
# stdp_comp.show()
display_component(current_mirror, scale = 1,path=".")

In [ ]:
# =========================================================================
# 4. ADD AVDD & AVSS POWER RAILS AND CONNECT SUPPLY NODES
# =========================================================================

# https://gemini.google.com/app/131a537106def8cd
# Calculate full horizontal span of the cell to draw supply rails
bbox = evaluate_bbox(current_mirror)
top_y = current_mirror.ymax + 2.0
bottom_y = current_mirror.ymin - 2.0

# Draw horizontal metal2 power rails
avdd_rail = current_mirror << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avdd_rail.move((-2, top_y))

avss_rail = current_mirror << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avss_rail.move((-2, bottom_y))

# Add global ports for the supply rails
current_mirror.add_port("avdd", center=(bbox[0]/2, top_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal2"), port_type="electrical")
current_mirror.add_port("avss", center=(bbox[0]/2, bottom_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal2"), port_type="electrical")
# add label to those global ports 
current_mirror.add_label(text="avdd", position=(bbox[0]/2, top_y + 0.5), layer=pdk.get_glayer("met2_label") , magnification=1.1)
current_mirror.add_label(text="avss", position=(bbox[0]/2, bottom_y + 0.5), layer=pdk.get_glayer("met2_label") , magnification=1.1)


In [ ]:
# stdp_comp.show()
display_component(current_mirror, scale = 1,path=".")

In [ ]:
#Adding VIAS
viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.
node_A_via = current_mirror << viam2m3
node_A_via_2 = current_mirror << viam2m3

node_A_via.move(m4_ref.ports["gate_S"].center).movex(2.8).movey(-5)
node_A_via_2.move(m4_ref.ports["gate_S"].center).movex(-2.5).movey(-5)

node_B_via = current_mirror << viam2m3
node_B_via.move(m3_ref.ports["gate_S"].center).movex(2.8).movey(-2)


In [ ]:
# =========================================================================
# 5. ADD MET2 PHYSICAL PINS AND LABELS FOR LVS
# =========================================================================

# Map port names to their source port objects
pins_to_create = {
    "A": node_A_via.ports["top_met_N"],
    "B": node_B_via.ports["top_met_N"]
}

for pin_name, port in pins_to_create.items():
    # Extract center position, size, and orientation from port object
    center = port.center
    width = port.width
    height = port.width  # Standard square pin area around the port center

    # 1. Add pin rectangle shape on met2_pin
    pin_rect = current_mirror << gf.components.rectangle(
        size = (width,height),
        layer=pdk.get_layer("metal2")
    )
    pin_rect.move((center[0] - width , center[1] - height))
    
    current_mirror.add_port(pin_name, center=center, width=1.0, orientation=180, layer=pdk.get_layer("metal2"),  port_type="electrical") 

    # 2. Add text label on met2_pin layer for net identity
    current_mirror.add_label(
        text=pin_name, position=center, layer=pdk.get_glayer("met2_label"), magnification=1.2
    )
    


In [ ]:
# Connect PMOS bulks to avdd rail
current_mirror << L_route(pdk, m4_ref.ports["tie_W_top_met_N"], current_mirror.ports["avdd"])
current_mirror << L_route(pdk, m3_ref.ports["tie_W_top_met_N"], current_mirror.ports["avdd"])

# Connect NMOS BULKS TO AVSS
current_mirror << L_route(pdk, m1_ref.ports["tie_W_top_met_S"], current_mirror.ports["avss"])
current_mirror << L_route(pdk, m2_ref.ports["tie_W_top_met_S"], current_mirror.ports["avss"])

#connecting PMOS SOURCES TO AVDD
current_mirror << L_route(pdk,  m4_ref.ports["multiplier_0_source_E"], m4_ref.ports["tie_E_top_met_N"])  # SOURCE m4
current_mirror << L_route(pdk,  m3_ref.ports["multiplier_0_source_E"], m3_ref.ports["tie_E_top_met_N"])  # SOURCE M3

#CONNECTING NMOS SOURCE TO AVSS
current_mirror << L_route(pdk,  m1_ref.ports["multiplier_0_source_E"], m1_ref.ports["tie_E_top_met_N"]) #m1
current_mirror << L_route(pdk,  m2_ref.ports["multiplier_0_source_E"], m2_ref.ports["tie_E_top_met_N"]) #m2

In [ ]:
#NODE A

current_mirror << L_route(pdk, m4_ref.ports["drain_W"], node_A_via_2.ports["top_met_N"])
current_mirror << L_route(pdk, m1_ref.ports["drain_W"], node_A_via_2.ports["top_met_S"])
current_mirror << L_route(pdk, m4_ref.ports["gate_W"], node_A_via_2.ports["top_met_N"])
current_mirror << L_route(pdk, m1_ref.ports["gate_W"], node_A_via_2.ports["top_met_S"])
current_mirror << straight_route(pdk, node_A_via_2.ports["bottom_met_W"], node_A_via.ports["bottom_met_E"])
current_mirror << L_route(pdk, m2_ref.ports["gate_W"], node_A_via.ports["top_met_S"])

In [ ]:
#NODE B
current_mirror << c_route(pdk, m3_ref.ports["drain_E"], node_B_via.ports["bottom_met_E"])
current_mirror << c_route(pdk, m3_ref.ports["gate_E"], node_B_via.ports["bottom_met_E"])
current_mirror << c_route(pdk, m2_ref.ports["drain_W"], node_B_via.ports["bottom_met_W"])


In [ ]:
# stdp_comp.show()
display_component(current_mirror, scale = 4,path=".")

In [ ]:
current_mirror.pprint_ports()

In [ ]:
evaluate_bbox(current_mirror)

In [ ]:
current_mirror_est_area = evaluate_bbox(current_mirror)[0] * evaluate_bbox(current_mirror)[1]
current_mirror_est_area

In [ ]:
m1_ref.info['netlist'].generate_netlist()

In [ ]:
# netlist = Netlist(circuit_name="current_mirror", nodes=['avss', 'avdd', 'B', 'A'])
# netlist.connect_netlist(m1_ref.info['netlist'], [('D', 'A'),   ('G', 'A'),   ('S', 'avss'), ('B', 'avss')])
# netlist.connect_netlist(m2_ref.info['netlist'], [('D', 'B'),   ('G', 'A'),   ('S', 'avss'), ('B', 'avss')])
# netlist.connect_netlist(m3_ref.info['netlist'], [('D', 'B'),   ('G', 'B'),   ('S', 'avdd'), ('B', 'avdd')])
# netlist.connect_netlist(m4_ref.info['netlist'], [('D', 'A'),   ('G', 'A'),   ('S', 'avdd'), ('B', 'avdd')])

# current_mirror.info["netlist"] = netlist
# print(current_mirror.info['netlist'].generate_netlist())

In [ ]:
import os
from pathlib import Path
import tempfile

path_to_dir = "/foss/designs/libs/snn_analog/current_mirror/"
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"


pex_path = path_to_dir + f"{current_mirror.name}.spice"
gds_path = path_to_dir + f"{current_mirror.name}.gds"
current_mirror.write_gds(str(gds_path))


magic_script_content = f"""
drc off            
gds flatglob *\\$\\$*
gds read {gds_path}
flatten {current_mirror.name}
load {current_mirror.name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {str(pex_path)}
exit
"""

with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
    magic_script_file.write(magic_script_content)
    magic_script_path = magic_script_file.name
    
magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
magic_subproc = subprocess.run(
    magic_cmd, 
    shell=True,
    check=True,
    capture_output=True
)

magic_subproc_code = magic_subproc.returncode
magic_subproc_out = magic_subproc.stdout.decode('utf-8')
print(magic_subproc_out)




In [ ]:
drc_result = gf180.drc_magic(current_mirror, current_mirror.name)

In [ ]:
import glob
extensions = [
            "els"
            "*.gds",
            "*.ext",
            "*.res.ext",
            "*.lvs.rpt",
            "*_lvs.rpt",
            "*.nodes",
            "*.sim",
            "*.pex.spice",
            "*_pex.spice"
            ]
files_to_delete = []
for ext in extensions:
    files_to_delete.extend(glob.glob(ext))
    
# Delete the files
for file_path in files_to_delete:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

In [ ]:


gf180.lvs_netgen(
    layout=current_mirror,
    design_name = current_mirror.name,
    pdk_root = Path(os.environ['PDKPATH']),
    lvs_setup_tcl_file = Path(os.environ['PDKPATH']) / "libs.tech" / "netgen" / f"{os.environ['PDK']}_setup.tcl",
    netlist = Path(str(path_to_dir + "current_mirror_sch.spice")),
    output_file_path =  Path(str(path_to_dir))
)
